# Finite Differences, Numerical Integration, and Heun's Method

Four independent problems: comparing finite-difference derivative approximations of increasing
order, recovering velocity/acceleration from discrete position data, comparing numerical
integration schemes (rectangles, Simpson's 1/3, Monte Carlo), and solving an ODE with Heun's
method (an explicit predictor-corrector / improved Euler scheme).

### 1. Finite-difference derivative approximations of increasing order

a) First- vs. second-order forward-difference derivative of `f(x) = x cos(x) sin(sqrt(1+x^2)/2)` at
`x = pi`, compared against a high-accuracy reference.

In [ ]:
import numpy as np

x1 = np.pi  # point where the derivative is evaluated

# Target function
def f(x):
    return x * np.cos(x) * np.sin(np.sqrt(1 + x**2) / 2)

# First-order forward-difference derivative
def order_one(x, dx):
    return (-3*f(x) + 4*f(x+dx) - f(x+2*dx)) / (2*dx)

# Second-order forward-difference derivative
def order_two(x, dx):
    return (-11*f(x) + 18*f(x+dx) - 9*f(x+2*dx) + 2*f(x+3*dx)) / (6*dx)

# 1. Get the "true value" using second order and h = 0.0001
dx_true_value = 0.0001
true_value = order_two(x1, dx_true_value)
print(f"\nValue taken as true (order 2, h = 0.0001): {true_value:.10f}\n")

# 2. Compute the derivative with h = 1 using order 1 and 2
h = 1

value_order1 = order_one(x1, h)
abs_error_order1 = np.abs(value_order1 - true_value)

value_order2 = order_two(x1, h)
abs_error_order2 = np.abs(value_order2 - true_value)

print(f" [Order 1, h = 1]")
print(f"Approximate value: {value_order1:.10f}")
print(f"Absolute error:    {abs_error_order1:.10f}\n")

print(f" [Order 2, h = 1]")
print(f"Approximate value: {value_order2:.10f}")
print(f"Absolute error:    {abs_error_order2:.10f}")


In [ ]:
import numpy as np
import pandas as pd

# Point where the derivative is evaluated
x1 = np.pi

# Target function
def f(x):
    return x * np.cos(x) * np.sin(np.sqrt(1 + x**2) / 2)

# Corrected derivatives of different orders
def order_one(x, dx):
    return (-3*f(x) + 4*f(x+dx) - f(x+2*dx)) / (2*dx)

def order_two(x, dx):
    return (-11*f(x) + 18*f(x+dx) - 9*f(x+2*dx) + 2*f(x+3*dx)) / (6*dx)

def order_three(x, dx):
    return (-25*f(x) + 48*f(x+dx) - 36*f(x+2*dx) + 16*f(x+3*dx) - 3*f(x+4*dx)) / (12*dx)

# "True" reference value using second order and a small h
true_value = order_two(x1, 0.0001)

# h values to evaluate
h_vals = [1, 0.1, 0.01]

# Results table
results = []

for h in h_vals:
    d1 = order_one(x1, h)
    d2 = order_two(x1, h)
    d3 = order_three(x1, h)

    results.append({
        "h": h,
        "Derivative order 1": d1,
        "Error order 1": abs(d1 - true_value),
        "Derivative order 2": d2,
        "Error order 2": abs(d2 - true_value),
        "Derivative order 3": d3,
        "Error order 3": abs(d3 - true_value)
    })

# Show the results table
table = pd.DataFrame(results)
print(f"Value taken as true (order 2, h = 0.0001): {true_value:.10f}\n")
print(table)


b) Same function, comparing a first-order approximation at a coarse step against a third-order
reference:

In [ ]:
import numpy as np

# Point where the derivative is evaluated
x1 = np.pi

# Target function
def f(x):
    return x * np.cos(x) * np.sin(np.sqrt(1 + x**2) / 2)

# First-order derivative
def order_one(x, dx):
    return (-3*f(x) + 4*f(x + dx) - f(x + 2*dx)) / (2*dx)

# Third-order derivative
def order_three(x, dx):
    return (-25*f(x) + 48*f(x + dx) - 36*f(x + 2*dx) + 16*f(x + 3*dx) - 3*f(x + 4*dx)) / (12*dx)

# Compute the true value using order 3 and h = 0.01
h_true_value = 0.01
true_value = order_three(x1, h_true_value)

# Compute the first-order derivative with h = 1
h_approx = 1
approx_value = order_one(x1, h_approx)

# Compute the absolute error
absolute_error = np.abs(approx_value - true_value)

# Show results
print(f"Value taken as true (order 3, h = 0.01): {true_value:.10f}")
print(f"Approximate value (order 1, h = 1):      {approx_value:.10f}")
print(f"Absolute error:                           {absolute_error:.10f}")


### 2. Velocity and acceleration from discrete position data

Given position samples at six evenly-spaced times, recover velocity and acceleration via
finite-difference formulas (forward/backward at the endpoints, centered in the interior).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Given data
t = np.array([0, 1, 2, 3, 4, 5])  # time in seconds
x = np.array([0, 2, 8, 18, 32, 50])  # position in meters

# Initialize arrays for velocity and acceleration
v = np.zeros_like(x, dtype=float)
a = np.zeros_like(x, dtype=float)

# Compute velocity (first derivative)
v[0] = (x[1] - x[0]) / (t[1] - t[0])  # forward
v[-1] = (x[-1] - x[-2]) / (t[-1] - t[-2])  # backward
for i in range(1, len(x)-1):
    v[i] = (x[i+1] - x[i-1]) / (t[i+1] - t[i-1])  # centered

# Compute acceleration (second derivative)
dt = t[1] - t[0]  # uniform time step
a[0] = (x[2] - 2*x[1] + x[0]) / dt**2
a[-1] = (x[-1] - 2*x[-2] + x[-3]) / dt**2
for i in range(1, len(x)-1):
    a[i] = (x[i+1] - 2*x[i] + x[i-1]) / dt**2

# Build a results DataFrame
table = pd.DataFrame({
    'Time (s)': t,
    'Position (m)': x,
    'Velocity (m/s)': v,
    'Acceleration (m/s^2)': a
})

# Show the table
print(table)

# Plot all three quantities
plt.figure(figsize=(12, 8))

# Position
plt.subplot(3, 1, 1)
plt.plot(t, x, 'o-', label='Position (m)')
plt.ylabel('Position (m)')
plt.grid(True)
plt.legend()

# Velocity
plt.subplot(3, 1, 2)
plt.plot(t, v, 'o-', color='orange', label='Velocity (m/s)')
plt.ylabel('Velocity (m/s)')
plt.grid(True)
plt.legend()

# Acceleration
plt.subplot(3, 1, 3)
plt.plot(t, a, 'o-', color='green', label='Acceleration (m/s^2)')
plt.xlabel('Time (s)')
plt.ylabel('Acceleration (m/s^2)')
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()


### 3. Numerical integration: rectangles, Simpson's 1/3, and Monte Carlo

a) All three methods compared against `scipy.integrate.quad` for `f(x) = sin(sin(x))` on `[0, pi]`:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.integrate import quad

# Define the function
def f(x):
    return np.sin(np.sin(x))

# Integration interval
a = 0
b = np.pi 

# Reference value of the integral
true_value, _ = quad(f, a, b)

# Integration methods
def rectangle_method(f, a, b, n):
    dx = (b - a) / n
    return dx * sum(f(a + i*dx) for i in range(n))

def simpson_method(f, a, b, n):
    if n % 2 != 0:
        raise ValueError("n must be even for Simpson's 1/3 rule")
    dx = (b - a) / n
    x = np.linspace(a, b, n+1)
    y = f(x)
    return dx/3 * (y[0] + 4 * sum(y[1:n:2]) + 2 * sum(y[2:n-1:2]) + y[n])

def monte_carlo_method(f, a, b, n):
    x_vals = np.linspace(a, b, 1000)
    ymax = np.max(f(x_vals))
    total_area = (b - a) * ymax

    points_inside = 0
    for _ in range(n):
        x_rand = np.random.uniform(a, b)
        y_rand = np.random.uniform(0, ymax)
        if y_rand <= f(x_rand):
            points_inside += 1

    proportion = points_inside / n
    return total_area * proportion

# Evaluate for different n
n_vals = [20]
results = []

for n in n_vals:
    riemann = rectangle_method(f, a, b, n)
    simpson = simpson_method(f, a, b, n)
    monte = monte_carlo_method(f, a, b, n)

    results.append({
        "n": n,
        "Rectangles": riemann,
        "Simpson": simpson,
        "Monte Carlo": monte,
        "Error Rectangles": abs(riemann - true_value),
        "Error Simpson": abs(simpson - true_value),
        "Error Monte Carlo": abs(monte - true_value)
    })

# Show results as a table
df = pd.DataFrame(results)
print(df)

# Plot the function
x_plot = np.linspace(a, b, 500)
y_plot = f(x_plot)

plt.figure(figsize=(10, 6))
plt.plot(x_plot, y_plot, color="blue")
plt.title("Integrand function")
plt.xlabel("x")
plt.ylabel("f(x)")
plt.grid(True)
plt.legend()
plt.show()


b) Coarse rectangle-rule approximation (n=4) checked against a finer Simpson reference (n=20):

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Function
def f(x):
    return np.sin(np.sin(x))

# Interval
a = 0
b = np.pi

# Rectangle method
def rectangle_method(f, a, b, n):
    dx = (b - a) / n
    return dx * sum(f(a + i*dx) for i in range(n))

# Simpson's 1/3 method
def simpson_method(f, a, b, n):
    if n % 2 != 0:
        raise ValueError("n must be even for Simpson's 1/3 rule")
    dx = (b - a) / n
    x = np.linspace(a, b, n+1)
    y = f(x)
    return dx/3 * (y[0] + 4 * sum(y[1:n:2]) + 2 * sum(y[2:n-1:2]) + y[n])

# Value taken as "true" via Simpson with n=20
n_simpson = 20
true_value = simpson_method(f, a, b, n_simpson)

# Rectangle approximation with n=4
n_rect = 4
rectangle_value = rectangle_method(f, a, b, n_rect)
rectangle_error = abs(true_value - rectangle_value)

# Show results
print(f"Value taken as true (Simpson, n=20): {true_value:.10f}")
print(f"Value via rectangles (n=4):          {rectangle_value:.10f}")
print(f"Absolute error:                       {rectangle_error:.10f}")


c) All three methods compared against each other (Monte Carlo as reference):

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Function to integrate
def f(x):
    return np.sin(np.sin(x))

# Interval
a = 0
b = np.pi

# Rectangle method
def rectangle_method(f, a, b, n):
    dx = (b - a) / n
    return dx * sum(f(a + i*dx) for i in range(n))

# Simpson's 1/3 method
def simpson_method(f, a, b, n):
    if n % 2 != 0:
        raise ValueError("n must be even for Simpson's 1/3 rule")
    dx = (b - a) / n
    x = np.linspace(a, b, n + 1)
    y = f(x)
    return dx / 3 * (y[0] + 4 * sum(y[1:n:2]) + 2 * sum(y[2:n-1:2]) + y[n])

# Monte Carlo method
def monte_carlo_method(f, a, b, n):
    x_vals = np.linspace(a, b, 1000)
    ymax = np.max(f(x_vals))
    total_area = (b - a) * ymax
    points_inside = 0

    for _ in range(n):
        x_rand = np.random.uniform(a, b)
        y_rand = np.random.uniform(0, ymax)
        if y_rand <= f(x_rand):
            points_inside += 1

    proportion = points_inside / n
    return total_area * proportion

# n for each method
n_deterministic = 20
n_montecarlo = 1000

# Results
rect_value = rectangle_method(f, a, b, n_deterministic)
simpson_value = simpson_method(f, a, b, n_deterministic)
monte_value = monte_carlo_method(f, a, b, n_montecarlo)

# Relative errors vs. Monte Carlo
rel_error_rect = abs((rect_value - monte_value) / monte_value)
rel_error_simpson = abs((simpson_value - monte_value) / monte_value)

# Show results as a table
df = pd.DataFrame([
    {
        "Method": "Rectangles",
        "Approximate value": rect_value,
        "Relative error vs. Monte Carlo": rel_error_rect
    },
    {
        "Method": "Simpson",
        "Approximate value": simpson_value,
        "Relative error vs. Monte Carlo": rel_error_simpson
    },
    {
        "Method": "Monte Carlo (reference)",
        "Approximate value": monte_value,
        "Relative error vs. Monte Carlo": 0.0
    }
])

print(df)


### 4. Heun's method (explicit predictor-corrector) for an ODE

Solving `dy/dx = y x^2 - y` from `x=0` to `x=3`, `y(0)=1`, with step size `h=0.2`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Define the function f(x, y) = dy/dx
def f(x, y):
    return y * x**2 - y

# Heun's method implementation
def heun_method(f, x0, y0, h, xf):
    x_vals = [x0]
    y_vals = [y0]
    n = int((xf - x0) / h)

    for _ in range(n):
        x = x_vals[-1]
        y = y_vals[-1]

        # Predictor step (Euler)
        y_euler = y + h * f(x, y)
        x_next = x + h

        # Corrector step (Heun)
        y_next = y + (h / 2) * (f(x, y) + f(x_next, y_euler))

        # Append new values
        x_vals.append(x_next)
        y_vals.append(y_next)

    return x_vals, y_vals

# Parameters
x0 = 0
xf = 3
y0 = 1
h = 0.2

# Run the method
x_vals, y_vals = heun_method(f, x0, y0, h, xf)

# Build a results table
table = pd.DataFrame({
    'x': x_vals,
    'y (approx)': y_vals
})

# Show the table
print(table)

# Plot the solution
plt.figure(figsize=(8, 5))
plt.plot(x_vals, y_vals, 'o-', color='blue', label='Heun (h=0.2)')
plt.title("Approximate Solution via Heun's Method")
plt.xlabel('x')
plt.ylabel('y(x)')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()
